# Visualize a query niche

Set `source_slice` and `center_cell_name`, run all cells, and the notebook will display the same **Selected niche on the source slice** figure used by `report.html`. The existing visualization script creates the figure and performs the same validation as the HTML report.

In [ ]:
from pathlib import Path
import csv
import json
import scanpy as sc
import matplotlib.pyplot as plt

# Inputs: edit these two values.
source_slice = "C57BL6J-638850.28"
center_cell_name = "1019171910102020454"

# Repository-relative paths for the slice and exported niche metrics.
repo = Path.cwd()
if not (repo / "experiment/visualize_query_niche.py").is_file():
    repo = repo.parent
assert (repo / "experiment/visualize_query_niche.py").is_file(), "Run this notebook from the repository or its notebooks directory."
metrics_path = repo / "experiment/query_niche_metrics/preprocessed/large" / f"{source_slice}.csv"
h5ad_path = repo / "data/20260601_225717" / f"{source_slice}.h5ad"
with metrics_path.open(newline="", encoding="utf-8") as handle:
    row = next((r for r in csv.DictReader(handle) if r["center_cell_name"] == str(center_cell_name)), None)
assert row is not None, f"Center cell not found: {center_cell_name}"
niche_cells = set(json.loads(row["niche_cell_names"]))
adata = sc.read_h5ad(h5ad_path)
cell_names = adata.obs_names.astype(str)
assert "spatial" in adata.obsm, f"No 'spatial' coordinates; available keys: {list(adata.obsm.keys())}"
coords = adata.obsm["spatial"][:, :2]
is_niche = cell_names.isin(niche_cells)
is_center = cell_names == str(center_cell_name)
assert is_center.any() and is_niche[is_center].all(), "Center is missing from the source slice or niche"
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
for ax in axes:
    ax.scatter(coords[:, 0], coords[:, 1], s=1, c="#d9dee7", linewidths=0, rasterized=True)
    ax.scatter(coords[is_niche, 0], coords[is_niche, 1], s=3, c="#e45756", linewidths=0, rasterized=True)
    ax.scatter(coords[is_center, 0], coords[is_center, 1], s=70, c="#174a8b", edgecolors="white", linewidths=0.8, zorder=3)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
axes[0].set_title(f"Full slice: {source_slice}")
niche_min, niche_max = coords[is_niche].min(axis=0), coords[is_niche].max(axis=0)
padding = max((niche_max - niche_min).max() * 0.15, 1)
axes[1].set_xlim(niche_min[0] - padding, niche_max[0] + padding)
axes[1].set_ylim(niche_min[1] - padding, niche_max[1] + padding)
axes[1].set_title(f"Niche close-up: {len(niche_cells):,} cells")
fig.suptitle(f"Selected niche on the source slice\nCenter cell: {center_cell_name}")
plt.show()

The figure is rendered directly from Scanpy and Matplotlib; no report or image files are written.